In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import make_scorer, accuracy_score, f1_score
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from imblearn.combine import SMOTEENN




# Suport functions

In [10]:
def confusion(true, pred):
    """
    Function for pretty printing confusion matrices
    """
    true.name = 'target'
    pred.name = 'predicted'
    cm = pd.crosstab(true.reset_index(drop=True), pred.reset_index(drop=True))
    cm = cm[cm.index]
    return cm

# Data loading

In [11]:
ILDS = pd.read_csv("preprocess_train_v4.csv", delimiter=',', header = None)

ILDS.columns = ['Age', 'ALB', 'AR', 'DBRatio', 'ALBIScore', 'Glob', 'LogTB', 'LogAlkphos', 'LogSgpt', 'LogSgot', 'Female', 'Target']


display(ILDS)

,Age,ALB,AR,DBRatio,ALBIScore,Glob,LogTB,LogAlkphos,LogSgpt,LogSgot,Female,Target
0,48,2.4,0.52,0.511111,0.226640,4.615385,1.504077,5.641907,2.564949,4.304065,0,0
1,39,4.3,1.38,0.473684,-0.182383,3.115942,0.641854,5.192957,3.737670,4.127134,0,0
2,23,3.1,1.00,0.300000,-0.264120,3.100000,0.000000,5.356586,3.713572,4.382027,0,0
3,42,3.2,1.06,0.285714,-0.374875,3.018868,-0.356675,5.023881,3.555348,4.394449,1,0
4,54,3.4,0.80,0.504425,0.604032,4.250000,3.117950,6.324359,3.401197,3.610918,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
445,38,3.1,1.03,0.250000,-0.328081,3.009709,-0.223144,4.976734,2.944439,3.135494,1,1
446,63,3.9,1.85,0.222222,-0.362480,2.108108,-0.105361,5.267858,3.951244,3.806662,0,1
447,60,3.5,1.00,0.285714,-0.400435,3.500000,-0.356675,5.141664,3.433987,3.258097,0,1
448,35,3.6,1.20,0.222222,-0.336920,3.000000,-0.105361,5.247024,3.218876,2.995732,0,1


In [12]:
X = ILDS.loc[:, ILDS.columns != 'Target']
y = ILDS['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify = y, random_state=1234)

In [13]:
results_df = pd.DataFrame(index=[], columns= ['Accuracy', 'F1 Macro', 'Precision Macro', 'Recall Macro'])

# Decision tree

# Random forest

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=1234, class_weight='balanced')
comb = SMOTEENN(random_state=42)
X_res, y_res = comb.fit_resample(X_train, y_train)
#rf.fit(X_res, y_res)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

In [15]:
confusion(y_train, pd.Series(rf.predict(X_train)))

predicted,0,1
target,,
0,258,0
1,0,102


In [16]:
confusion(y_test, pd.Series(rf.predict(X_test)))     

predicted,0,1
target,,
0,61,3
1,21,5


In [17]:
cross_val_results = pd.DataFrame(cross_validate(rf, X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['RF',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.719444,0.582799,0.659556,0.584204


# Gradient Boosting

In [18]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

weights = compute_sample_weight(class_weight='balanced', y=y_train)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train, sample_weight=weights)
gb_pred = gb.predict(X_test)

In [19]:
cross_val_results = pd.DataFrame(cross_validate(gb , X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['GB',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.719444,0.582799,0.659556,0.584204
GB,0.7,0.584447,0.618648,0.588063


# Voting classifier

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier

log_clf = LogisticRegression(class_weight="balanced", max_iter=1000)
svc_clf = SVC(probability=True, class_weight="balanced", max_iter=1000)
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
qda = QuadraticDiscriminantAnalysis()
voting = VotingClassifier(estimators=[
    ('lr', log_clf), ('svc', svc_clf), ('rf', rf), ('qda', qda)
], voting='soft')

comb = SMOTEENN(random_state=42)
X_res, y_res = comb.fit_resample(X_train, y_train)
#voting.fit(X_res, y_res)
voting.fit(X_train, y_train)
voting_pred = voting.predict(X_test)


C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\discriminant_analysis.py:935: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")


In [21]:
confusion(y_train, pd.Series(voting.predict(X_train)))

predicted,0,1
target,,
0,228,30
1,26,76


In [22]:
confusion(y_test, pd.Series(voting.predict(X_test)))

predicted,0,1
target,,
0,53,11
1,15,11


In [23]:
from sklearn.model_selection import train_test_split

# Suppose you start with X and y:
#   X_full: all feature vectors
#   y_full: all labels

# 1) Split off 20% for a hold‑out test set:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_test, y_test, test_size=0.2, random_state=42, stratify=y_test
)

# 2) From the remaining 80%, split again: 80% → train, 20% → validation.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)


In [24]:
import numpy as np
from sklearn.metrics import f1_score

probs = voting.predict_proba(X_val)[:, 1]
best_thresh, best_f1 = 0.5, 0
for t in np.linspace(0.1, 0.9, 41):
    f1 = f1_score(y_val, (probs >= t).astype(int))
    if f1 > best_f1:
        best_f1, best_thresh = f1, t

print(f"Best F1={best_f1:.3f} at threshold={best_thresh:.2f}")
# Then use that threshold on test set:
y_pred = (voting.predict_proba(X_test)[:,1] >= best_thresh).astype(int)


Best F1=0.500 at threshold=0.20


In [25]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score

param_grid = {
    'rf__n_estimators': [100, 300],
    'rf__class_weight': [None, 'balanced'],
    'lr__C': [0.01, 1, 10],
    'svc__C': [0.1, 1, 10],
    'svc__gamma': ['scale', 'auto'],
}
grid = GridSearchCV(
    voting,
    param_grid,
    cv=5,
    scoring=make_scorer(f1_score),
    n_jobs=-1,
    verbose=2
)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV F1:", grid.best_score_)
voting = grid.best_estimator_


Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best params: {'lr__C': 0.01, 'rf__class_weight': None, 'rf__n_estimators': 100, 'svc__C': 0.1, 'svc__gamma': 'scale'}
Best CV F1: 0.18


C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\discriminant_analysis.py:935: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")


In [26]:
log_clf = LogisticRegression(class_weight="balanced", max_iter=1000, C=10)
svc_clf = SVC(probability=True, class_weight="balanced", max_iter=1000, C=0.1, gamma='scale')
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
qda = QuadraticDiscriminantAnalysis()
voting = VotingClassifier(estimators=[
    ('lr', log_clf), ('svc', svc_clf), ('rf', rf), ('qda', qda)
], voting='soft')

In [27]:
cross_val_results = pd.DataFrame(cross_validate(voting , X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['CLF',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\discriminant_analysis.py:935: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\discriminant_analysis.py:935: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.719444,0.582799,0.659556,0.584204
GB,0.7,0.584447,0.618648,0.588063
CLF,0.684848,0.474897,0.442172,0.529167


# XGBoost

In [28]:
from xgboost import XGBClassifier

xgb = XGBClassifier(scale_pos_weight=1/2)
xgb.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [29]:
cross_val_results = pd.DataFrame(cross_validate(xgb , X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['XGB',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.719444,0.582799,0.659556,0.584204
GB,0.7,0.584447,0.618648,0.588063
CLF,0.684848,0.474897,0.442172,0.529167
XGB,0.633333,0.530582,0.522565,0.545833


In [30]:
confusion(y_train, pd.Series(xgb.predict(X_train)))

predicted,0,1
target,,
0,40,0
1,2,15


In [31]:
confusion(y_test, pd.Series(xgb.predict(X_test)))

predicted,0,1
target,,
0,10,3
1,1,4


# AdaBoosting

In [32]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier


In [33]:
base = DecisionTreeClassifier(max_depth=1, class_weight='balanced')
ada = AdaBoostClassifier(estimator=base, n_estimators=100, random_state=42)
ada.fit(X_train, y_train)


C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


AdaBoostClassifier(estimator=DecisionTreeClassifier(class_weight='balanced',
                                                    max_depth=1),
                   n_estimators=100, random_state=42)

In [34]:
cross_val_results = pd.DataFrame(cross_validate(ada , X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['ADA',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent th

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.719444,0.582799,0.659556,0.584204
GB,0.7,0.584447,0.618648,0.588063
CLF,0.684848,0.474897,0.442172,0.529167
XGB,0.633333,0.530582,0.522565,0.545833
ADA,0.631818,0.552425,0.600703,0.558333


In [35]:
confusion(y_train, pd.Series(ada.predict(X_train)))

predicted,0,1
target,,
0,40,0
1,0,17


In [36]:
confusion(y_test, pd.Series(ada.predict(X_test)))

predicted,0,1
target,,
0,11,2
1,3,2


# F1 Check

In [39]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall, prec, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

metrics_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

In [40]:
test_y = pd.read_csv("test_y.csv").iloc[:, 1]

ILDS_test = pd.read_csv("minimal_test_fs.csv", delimiter=',', header = None)

ILDS_test.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female']

random forest: rf

gradient boosting: gb

voting classifier: voting

xgboost: xgb

ada boosting: ada

In [41]:
from sklearn.ensemble import RandomForestClassifier

voting.fit(X_train, y_train)

rf.fit(X_train, y_train)

labels_rf = pd.DataFrame(columns = ['ID', 'Label'])
labels_rf['Label'] = pd.DataFrame(rf.predict(ILDS_test))
labels_rf['ID'] = labels_rf.index + 1

confusion(test_y, labels_rf['Label'])

compute_metrics(test_y, labels_rf['Label'])

C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\discriminant_analysis.py:935: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")


ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- Alkphos
- BilRatio
- Sgot
- TB
Feature names seen at fit time, yet now missing:
- ALBIScore
- DBRatio
- Glob
- LogAlkphos
- LogSgot
- ...


# Final test export

In [ ]:
xgb.fit(X_train, y_train)

ILDS_test = pd.read_csv("minimal_test_fs.csv", delimiter=',', header = None)

ILDS_test.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female']

X_test = ILDS_test.loc[:,:'Female']

ILDS_test['Label'] = rf.predict(X_test)


ILDS_test.index = ILDS_test.index + 1
ILDS_test.index.name = 'ID'

ILDS_test['Label'].to_csv('voting_classif_fs.csv', index=True)